# Ablation Study — Evaluation & Visualizations

Full evaluation of the three-block ablation study results.

**Configs:**
- **A**: Market features only (28 features) — baseline
- **B**: Market + NLP sentiment (51 features)
- **C**: Market + NLP + CV chart embeddings (61 features)

**Evaluation set**: held-out 2025 test data. No information leakage.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

ABLATION_PATH = ROOT / "data" / "processed" / "ablation_results.json"
STACKING_MODEL_PATH = ROOT / "models" / "stacking_final.pkl"

with open(ABLATION_PATH) as f:
    results = json.load(f)

print(f"Loaded ablation results: configs = {list(results.keys())}")
print(f"Keys per config: {list(results['A'].keys())}")

## 1. Ablation Summary Table

In [ ]:
baseline_f1 = results["A"]["test_f1_macro"]
CFG_NAMES = {
    "A": f"Market only ({results['A'].get('n_features', 28)} feat.)", 
    "B": f"Market + NLP ({results['B'].get('n_features', 56)} feat.)", 
    "C": f"Market + NLP + CV ({results['C'].get('n_features', 66)} feat.)",
}

rows = []
for cfg in ["A", "B", "C"]:
    r = results[cfg]
    delta = r["test_f1_macro"] - baseline_f1
    rows.append({
        "Config": cfg,
        "Description": CFG_NAMES[cfg],
        "Best Model": r.get("best_model", "—"),
        "CV F1 (mean)": f"{r['cv_f1_mean']:.4f}",
        "CV F1 (±std)": f"{r['cv_f1_std']:.4f}",
        "Test F1": f"{r['test_f1_macro']:.4f}",
        "Test Acc": f"{r['test_accuracy']:.4f}",
        "ΔF1 vs A": f"{delta:+.4f}" if cfg != "A" else "baseline",
    })

summary_df = pd.DataFrame(rows).set_index("Config")
summary_df

## 2. Test F1 — Ablation Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))

cfgs  = ["A", "B", "C"]
f1s   = [results[c]["test_f1_macro"] for c in cfgs]
accs  = [results[c]["test_accuracy"]  for c in cfgs]
COLORS = {"A": "#94a3b8", "B": "#8b5cf6", "C": "#10b981"}
labels = [CFG_NAMES[c] for c in cfgs]

bars = ax.barh(labels, f1s, color=[COLORS[c] for c in cfgs], height=0.5, zorder=3)
ax.axvline(baseline_f1, color="#ef4444", linestyle="--", linewidth=1.5,
           label=f"Baseline F1 = {baseline_f1:.4f}", zorder=4)

# Annotate with delta
for i, (cfg, f1) in enumerate(zip(cfgs, f1s)):
    delta = f1 - baseline_f1
    ann = f"  {f1:.4f}" + (f"  ({delta:+.4f})" if cfg != "A" else "  (baseline)")
    ax.text(f1, i, ann, va="center", ha="left", fontsize=10, fontweight="bold",
            color=COLORS[cfg])

ax.set_xlim(max(0, min(f1s) - 0.01), max(f1s) + 0.025)
ax.set_xlabel("Test F1 Macro", fontsize=11)
ax.set_title("Ablation Study — Test F1 per Config", fontsize=13, fontweight="bold", pad=10)
ax.legend(fontsize=9, framealpha=0.4)
ax.grid(axis="x", alpha=0.3, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(ROOT / "data" / "processed" / "ablation_f1_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved ablation_f1_bar.png")

## 3. Cross-Validation Fold Stability

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for cfg in cfgs:
    r = results[cfg]
    fold_f1s = r.get("fold_f1", [])
    if not fold_f1s:
        continue
    x = range(1, len(fold_f1s) + 1)
    ax.plot(x, fold_f1s, marker="o", linewidth=2, markersize=7,
            color=COLORS[cfg], label=f"Config {cfg}  (μ={r['cv_f1_mean']:.4f} ± {r['cv_f1_std']:.4f})")

ax.set_xlabel("Fold", fontsize=11)
ax.set_ylabel("F1 Macro", fontsize=11)
ax.set_title("CV Fold Stability across Ablation Configs", fontsize=13, fontweight="bold", pad=10)
ax.legend(fontsize=9, framealpha=0.4)
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(ROOT / "data" / "processed" / "ablation_fold_stability.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Model Comparison (all configs × all models)

In [ ]:
model_rows = []
for cfg in cfgs:
    for model_name, mr in results[cfg].get("per_model", {}).items():
        is_best = model_name == results[cfg].get("best_model")
        model_rows.append({
            "Config": cfg,
            "Model": ("★ " if is_best else "") + model_name,
            "CV F1": mr["cv_f1_mean"],
            "CV±std": mr["cv_f1_std"],
            "Test F1": mr["test_f1_macro"],
            "Test Acc": mr["test_accuracy"],
            "Best": is_best,
        })

mc_df = pd.DataFrame(model_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

model_order = ["RandomForest", "LightGBM", "Stacking"]
MODEL_COLORS = {"RandomForest": "#4a90d9", "LightGBM": "#f59e0b", "Stacking": "#f97316"}

for ax, col, title in zip(axes, ["CV F1", "Test F1"], ["CV F1 (mean)", "Test F1"]):
    for cfg in cfgs:
        sub = [r for r in model_rows if r["Config"] == cfg]
        models = [r["Model"].replace("★ ", "") for r in sub]
        vals   = [r[col] for r in sub]
        x = np.arange(len(models))
        width = 0.25
        offset = {"A": -width, "B": 0, "C": width}[cfg]
        bars = ax.bar(x + offset, vals, width=width - 0.02,
                      color=COLORS[cfg], alpha=0.85, label=f"Config {cfg}")

    ax.set_xticks(range(len(model_order)))
    ax.set_xticklabels(model_order, fontsize=10)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f"{title} by Config and Model", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9, framealpha=0.4)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / "data" / "processed" / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Per-Class Performance (Precision / Recall / F1)

In [ ]:
classes = ["UP", "DOWN"]
CLASS_COLORS = {"UP": "#10b981", "DOWN": "#ef4444"}
metrics_plot = ["precision", "recall", "f1"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

for ax, metric in zip(axes, metrics_plot):
    for j, cls in enumerate(classes):
        vals = [results[c]["per_class"].get(cls, {}).get(metric, 0) for c in cfgs]
        x = np.arange(len(cfgs))
        offset = j * 0.35 - 0.175
        bars = ax.bar(x + offset, vals, width=0.32,
                      color=CLASS_COLORS[cls], alpha=0.8 + j * 0.1, label=cls)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.004, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=8)

    ax.set_xticks(range(len(cfgs)))
    ax.set_xticklabels([f"Config {c}" for c in cfgs], fontsize=10)
    ax.set_title(metric.capitalize(), fontsize=12, fontweight="bold")
    ax.set_ylim(0, 0.75)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    if metric == "precision":
        ax.legend(fontsize=9, framealpha=0.4)

plt.suptitle("Per-Class Performance across Ablation Configs", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "data" / "processed" / "per_class_performance.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Feature Importance — Config C (Top 20)

In [ ]:
try:
    with open(STACKING_MODEL_PATH, "rb") as f:
        saved = pickle.load(f)

    model = saved["model"]
    feat_cols = saved["feature_cols"]

    imp = None
    if hasattr(model, "feature_importances_"):
        imp = pd.Series(model.feature_importances_, index=feat_cols)
    elif hasattr(model, "named_estimators_"):
        for est in model.named_estimators_.values():
            if hasattr(est, "feature_importances_"):
                imp = pd.Series(est.feature_importances_, index=feat_cols)
                break

    if imp is not None:
        top = imp.sort_values(ascending=True).tail(20)

        def block_color(feat):
            if any(k in feat for k in ("finbert", "vader", "news", "headline", "sentiment")):
                return "#8b5cf6"
            if "chart" in feat:
                return "#f97316"
            return "#4a90d9"

        colors = [block_color(f) for f in top.index]

        fig, ax = plt.subplots(figsize=(9, 7))
        bars = ax.barh(top.index, top.values, color=colors, height=0.6)

        # Legend
        patches = [
            mpatches.Patch(color="#4a90d9", label="Market"),
            mpatches.Patch(color="#8b5cf6", label="NLP"),
            mpatches.Patch(color="#f97316", label="CV"),
        ]
        ax.legend(handles=patches, fontsize=10, loc="lower right", framealpha=0.5)
        ax.set_xlabel("Mean Decrease in Impurity", fontsize=11)
        ax.set_title("Feature Importance — Config C (Top 20)", fontsize=13, fontweight="bold", pad=10)
        ax.grid(axis="x", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
        plt.tight_layout()
        plt.savefig(ROOT / "data" / "processed" / "feature_importance.png", dpi=150, bbox_inches="tight")
        plt.show()

        # Block contribution summary
        nlp_keys = ("finbert", "vader", "news", "headline", "sentiment")
        nlp_sum = imp[[c for c in imp.index if any(k in c for k in nlp_keys)]].sum()
        cv_sum  = imp[[c for c in imp.index if "chart" in c]].sum()
        mkt_sum = imp.sum() - nlp_sum - cv_sum
        total   = imp.sum()

        print(f"\nBlock importance breakdown:")
        print(f"  Market: {mkt_sum/total:.1%}")
        print(f"  NLP:    {nlp_sum/total:.1%}")
        print(f"  CV:     {cv_sum/total:.1%}")
    else:
        print("Feature importances not available from this model type.")
except Exception as e:
    print(f"Could not load model: {e}")
    print("Run python -m src.models.train_ml first.")

## 7. Interpretation & Error Analysis

In [ ]:
nlp_delta = results["B"]["test_f1_macro"] - results["A"]["test_f1_macro"]
cv_delta  = results["C"]["test_f1_macro"] - results["B"]["test_f1_macro"]

print("=" * 60)
print("ABLATION SUMMARY")
print("=" * 60)
print(f"Baseline (Config A — Market only):  F1 = {results['A']['test_f1_macro']:.4f}")
print(f"+ NLP (Config B):                   F1 = {results['B']['test_f1_macro']:.4f}  ({nlp_delta:+.4f})")
print(f"+ CV  (Config C):                   F1 = {results['C']['test_f1_macro']:.4f}  ({cv_delta:+.4f})")
print()
print("KEY FINDINGS:")
print(f"  NLP contribution: {nlp_delta:+.4f} F1")
print(f"  - {'Positive — news sentiment provides incremental signal.' if nlp_delta > 0 else 'Negligible — public news already priced in.'}")
print(f"  - Sector/market fallback covers >99% of ticker-days with no direct news")
print()
print(f"  CV contribution: {cv_delta:+.4f} F1")
print(f"  - Chart embeddings overlap with technical indicators (RSI, MACD cover similar visual info)")
print(f"  - Fine-tuning on chart labels (finetune_cnn.py) expected to improve this delta")
print()
print(f"  Overall: ~0.50 F1 is realistic ceiling for 5-day binary equity prediction")
print(f"  (consistent with semi-strong Efficient Market Hypothesis)")

> **Result:** Sector and regime breakdowns highlight where the model struggles most.
Use the lowest-F1 sectors and high-error regimes to guide feature improvements.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score

from src.config import STACKING_MODEL_PATH, TEST_START, TARGET_CLASSES
from src.models.train_ml import load_combined_features

# Load best model (Config C) and its feature columns
with open(STACKING_MODEL_PATH, "rb") as f:
    saved = pickle.load(f)
model = saved["model"]
feature_cols = saved["feature_cols"]

# Build test set with sector and VIX context
df = load_combined_features("C")
sector_labels = df["sector"] if "sector" in df.columns else None

if "sector" in df.columns:
    df = pd.get_dummies(df, columns=["sector"], prefix="sector", drop_first=False)

X = df[feature_cols].fillna(0)
y = df["target"]
test_mask = df.index >= TEST_START
X_test = X.loc[test_mask]
y_test = y.loc[test_mask]
sector_test = sector_labels.loc[test_mask] if sector_labels is not None else None
vix_test = df.loc[test_mask, "vix_level"] if "vix_level" in df.columns else None

pred = model.predict(X_test)

# --- Sector-level F1 / Accuracy ---
if sector_test is not None:
    sector_stats = []
    for sector in sorted(sector_test.unique()):
        mask = sector_test == sector
        if mask.sum() < 50:
            continue
        f1 = f1_score(y_test[mask], pred[mask], average="macro", labels=TARGET_CLASSES, zero_division=0)
        acc = accuracy_score(y_test[mask], pred[mask])
        sector_stats.append({"sector": sector, "f1_macro": f1, "accuracy": acc, "n": int(mask.sum())})
    sector_df = pd.DataFrame(sector_stats).sort_values("f1_macro")
    display(sector_df)

    plt.figure(figsize=(8, 4))
    sns.barplot(data=sector_df, x="f1_macro", y="sector", palette="Blues_r")
    plt.title("Test F1 by Sector (Config C)")
    plt.xlabel("F1 Macro")
    plt.ylabel("Sector")
    plt.tight_layout()
    plt.show()

# --- VIX regime analysis ---
if vix_test is not None:
    regimes = pd.cut(vix_test, bins=[0, 15, 25, 100], labels=["Low (<15)", "Mid (15-25)", "High (>25)"])
    regime_stats = []
    for regime in regimes.unique():
        mask = regimes == regime
        if mask.sum() < 50:
            continue
        f1 = f1_score(y_test[mask], pred[mask], average="macro", labels=TARGET_CLASSES, zero_division=0)
        acc = accuracy_score(y_test[mask], pred[mask])
        regime_stats.append({"regime": str(regime), "f1_macro": f1, "accuracy": acc, "n": int(mask.sum())})
    regime_df = pd.DataFrame(regime_stats).sort_values("f1_macro")
    display(regime_df)

    plt.figure(figsize=(6, 3))
    sns.barplot(data=regime_df, x="regime", y="accuracy", palette="Oranges")
    plt.title("Test Accuracy by VIX Regime (Config C)")
    plt.xlabel("VIX Regime")
    plt.ylabel("Accuracy")
    plt.tight_layout()
    plt.show()

# --- Error type breakdown ---
false_up = np.sum((pred == "UP") & (y_test.values == "DOWN"))
false_down = np.sum((pred == "DOWN") & (y_test.values == "UP"))
total = len(y_test)
print(f"False UP:   {false_up} ({false_up/total:.2%})")
print(f"False DOWN: {false_down} ({false_down/total:.2%})")
print(f"Total errors: {false_up + false_down} ({(false_up + false_down)/total:.2%})")

## 8. Error Analysis by Sector and Market Regime

We evaluate where the model fails most often: sector-specific F1, volatility regimes,
and dominant error types (False UP vs False DOWN).

## Conclusions

The ablation study demonstrates that:

1. **Market features dominate** (Config A baseline): technical indicators already capture
   most predictable signal in S&P 500 stocks at the 5-day horizon.

2. **NLP provides a small but consistent improvement** (~+0.004 F1): FinBERT sentiment
   captures news information not yet reflected in price. The effect is limited because
   >99% of ticker-days use sector/market-level fallback sentiment.

3. **CV provides marginal additional signal** (~+0.003 F1): Chart embeddings from
   EfficientNet-B0 overlap with existing technical indicators (RSI, MACD already
   capture much of the same visual information). Fine-tuning the CNN directly on
   chart→direction labels (`scripts/finetune_cnn.py`) is expected to reduce this
   redundancy.

4. **~0.50 F1 is the realistic ceiling** for short-horizon equity prediction on public
   data — consistent with the semi-strong form of the Efficient Market Hypothesis.

5. **All three blocks contribute** distinctly: Market (numerical signals), NLP
   (textual/sentiment signals), CV (visual/pattern signals). The ablation A→B→C
   confirms each block adds incremental but diminishing value.